# Deception Classifier — Colab GPU Training
Run each cell in order. Make sure the runtime is set to a GPU of your choice 
Runtime → Change runtime type → i.e., A100 GPU

## 1. Clone repo & install dependencies

In [1]:
!git clone https://github.com/Lucca878/deceptionClassifierTraining.git
%cd deceptionClassifierTraining

Cloning into 'deceptionClassifierTraining'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 213 (delta 38), reused 192 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (213/213), 4.47 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/deceptionClassifierTraining


In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 112.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.1 MB/s eta 0:00:00


## 2. Verify GPU

In [3]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: NVIDIA A100-SXM4-40GB


## 2.5 (Optional) Set hyperparameters to test
Set these BEFORE running CV. Leave as `None` to use defaults. Same parameters will be used for both CV and full training.

In [ ]:
# Set hyperparameters to test (None = use defaults)
EPOCHS = None          # int: number of epochs (default: 2)
LEARNING_RATE = None   # float: learning rate (default: 5e-5)
BATCH_SIZE = None      # int: batch size (default: 32)
WEIGHT_DECAY = None    # float: weight decay (default: 0.01)

# Example: to test with 3 epochs and higher LR:
# EPOCHS = 3
# LEARNING_RATE = 1e-4

print("Hyperparameters to test:")
print(f"  EPOCHS: {EPOCHS if EPOCHS is not None else '(default: 2)'}")
print(f"  LEARNING_RATE: {LEARNING_RATE if LEARNING_RATE is not None else '(default: 5e-5)'}")
print(f"  BATCH_SIZE: {BATCH_SIZE if BATCH_SIZE is not None else '(default: 32)'}")
print(f"  WEIGHT_DECAY: {WEIGHT_DECAY if WEIGHT_DECAY is not None else '(default: 0.01)'}")

Hyperparameters to test:
  EPOCHS: (default: 2)
  LEARNING_RATE: 4e-05
  BATCH_SIZE: (default: 32)
  WEIGHT_DECAY: (default: 0.01)


## 2.6 Select model

In [32]:
MODEL_KEY = "sbert"  # Options: distilbert, bert, sbert, modernbert
print(f"Selected model: {MODEL_KEY}")

Selected model: sbert


## 3. Run cross-validation first (tuning stage)

In [33]:
import os
import subprocess

# Build command with optional hyperparameters
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "cv",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running CV with command:")
print(" ".join(cmd))
print()

# Stream child process output live in notebook
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

Running CV with command:
python src/pipeline/run_pipeline.py --mode cv --model sbert --output_root ./outputs --seed 42 --lr 4e-05

Training model preset: sbert
Backbone: sentence-transformers/all-mpnet-base-v2
Loaded 4542 rows from data/hippocorpus/hippocorpus_training_truncated.csv

--- split_1 ---

Map: 100%|██████████| 3633/3633 [00:00<00:00, 7482.86 examples/s]

Map: 100%|██████████| 909/909 [00:00<00:00, 7568.20 examples/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3241.23it/s, Materializing param=mpnet.encoder.relative_attention_bias.weight]
MPNetForSequenceClassification LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                        | Status     | 
---------------------------+------------+-
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    |

## 4. Inspect CV results (decide hyperparameters/model)

In [34]:
import os, glob

# Find latest run that actually contains CV results
run_dirs = sorted(glob.glob(f"outputs/{MODEL_KEY}_*"))
cv_run_dirs = [d for d in run_dirs if os.path.exists(os.path.join(d, "cv_results.csv"))]
latest = cv_run_dirs[-1] if cv_run_dirs else None

print("Latest CV run:", latest)

if latest:
    for f in os.listdir(latest):
        print(" ", f)
else:
    print("No CV run found yet. Run section 3 first.")

Latest CV run: outputs/sbert_20260428_100832
  cv
  config.json
  cv_results.csv


In [35]:
import pandas as pd
import os
import glob
import json

cv_path = os.path.join(latest, "cv_results.csv") if latest else None
if cv_path and os.path.exists(cv_path):
    # File is semicolon-separated
    df = pd.read_csv(cv_path, sep=";")
    print(df.head())
    print("\nRows, cols:", df.shape)

    # Main CV metric from stored predictions
    mean_acc = df["Correct"].mean()
    print("Mean CV accuracy:", round(mean_acc, 4))

    # Optional per-split and per-class breakdown
    print("\nAccuracy by split:")
    print(df.groupby("Split")["Correct"].mean().round(4))

    print("\nRecall by label (0=truthful, 1=deceptive):")
    print(df.groupby("Label")["Correct"].mean().round(4))

    # Compact validation + train summary from trainer logs
    eval_rows = []
    overfit_flags = []
    for split in [f"split_{i}" for i in range(1, 6)]:
        pattern = os.path.join(latest, "cv", split, "**", "trainer_state.json")
        state_files = sorted(glob.glob(pattern, recursive=True))
        if not state_files:
            continue

        with open(state_files[-1], "r") as f:
            state = json.load(f)

        log_history = state.get("log_history", [])
        eval_logs = [x for x in log_history if "eval_loss" in x]
        train_logs = [x for x in log_history if "loss" in x and "epoch" in x]
        if not eval_logs:
            continue

        # Collect eval points and nearest train loss up to that epoch
        for log in eval_logs:
            eval_epoch = float(log.get("epoch")) if log.get("epoch") is not None else None
            train_candidates = [
                t for t in train_logs
                if t.get("epoch") is not None and eval_epoch is not None and float(t.get("epoch")) <= eval_epoch
            ]
            train_loss = float(train_candidates[-1].get("loss")) if train_candidates else None

            eval_rows.append({
                "EvalLoss": float(log.get("eval_loss")),
                "EvalAccuracy": float(log.get("eval_accuracy")),
                "TrainLoss": train_loss,
            })

        # Overfitting flag per fold: last eval loss worse than best eval loss
        losses = [float(log.get("eval_loss")) for log in eval_logs]
        overfit_flags.append(losses[-1] > min(losses))

    if eval_rows:
        eval_df = pd.DataFrame(eval_rows)
        mean_train_loss = eval_df["TrainLoss"].dropna().mean()
        train_part = f", mean_train_loss={mean_train_loss:.4f}" if pd.notna(mean_train_loss) else ""
        print(
            "\nCompact validation summary: "
            f"mean_eval_loss={eval_df['EvalLoss'].mean():.4f}, "
            f"mean_eval_accuracy={eval_df['EvalAccuracy'].mean():.4f}"
            f"{train_part}, "
            f"overfit_folds={sum(overfit_flags)}/{len(overfit_flags)}"
        )

     Split  Prediction  Label  Correct
0  split_1           0      1    False
1  split_1           1      0    False
2  split_1           1      0    False
3  split_1           1      0    False
4  split_1           1      1     True

Rows, cols: (4542, 4)
Mean CV accuracy: 0.7878

Accuracy by split:
Split
split_1    0.7844
split_2    0.7888
split_3    0.7896
split_4    0.7819
split_5    0.7941
Name: Correct, dtype: float64

Recall by label (0=truthful, 1=deceptive):
Label
0    0.7931
1    0.7828
Name: Correct, dtype: float64

Compact validation summary: mean_eval_loss=0.4852, mean_eval_accuracy=0.7777, mean_train_loss=0.4622, overfit_folds=0/5


In [36]:
import glob
import json
import os
import pandas as pd

# Pull fold-level eval + train metrics (per epoch) from Trainer logs
epoch_rows = []

if latest:
    for split in [f"split_{i}" for i in range(1, 6)]:
        # Look for trainer_state.json in split root and nested checkpoints
        pattern = os.path.join(latest, "cv", split, "**", "trainer_state.json")
        state_files = sorted(glob.glob(pattern, recursive=True))
        if not state_files:
            continue

        state_path = state_files[-1]
        with open(state_path, "r") as f:
            state = json.load(f)

        log_history = state.get("log_history", [])
        eval_logs = [x for x in log_history if "eval_loss" in x]
        train_logs = [x for x in log_history if "loss" in x and "epoch" in x]

        for log in eval_logs:
            eval_epoch = float(log.get("epoch")) if log.get("epoch") is not None else None
            train_candidates = [
                t for t in train_logs
                if t.get("epoch") is not None and eval_epoch is not None and float(t.get("epoch")) <= eval_epoch
            ]
            train_loss = float(train_candidates[-1].get("loss")) if train_candidates else None

            epoch_rows.append({
                "Split": split,
                "Epoch": eval_epoch,
                "TrainLoss": train_loss,
                "EvalLoss": float(log.get("eval_loss")) if log.get("eval_loss") is not None else None,
                "EvalAccuracy": float(log.get("eval_accuracy")) if log.get("eval_accuracy") is not None else None,
            })

if epoch_rows:
    epoch_df = pd.DataFrame(epoch_rows).sort_values(["Split", "Epoch"])

    print("Train/validation metrics per epoch by split:")
    print(epoch_df.to_string(index=False))

    # Overfitting summary: last eval loss worse than best eval loss while train loss did not increase
    summary_rows = []
    for split, g in epoch_df.groupby("Split"):
        g = g.sort_values("Epoch")
        best_idx = g["EvalLoss"].idxmin()
        best = g.loc[best_idx]
        last = g.iloc[-1]

        train_improved_or_flat = (
            pd.notna(best["TrainLoss"]) and pd.notna(last["TrainLoss"]) and last["TrainLoss"] <= best["TrainLoss"]
        )
        eval_worsened = bool(last["EvalLoss"] > best["EvalLoss"])
        overfit_flag = bool(eval_worsened and train_improved_or_flat)

        summary_rows.append({
            "Split": split,
            "BestEpoch": best["Epoch"],
            "BestTrainLoss": best["TrainLoss"],
            "BestEvalLoss": best["EvalLoss"],
            "LastEpoch": last["Epoch"],
            "LastTrainLoss": last["TrainLoss"],
            "LastEvalLoss": last["EvalLoss"],
            "DeltaTrain(Last-Best)": (
                (last["TrainLoss"] - best["TrainLoss"])
                if pd.notna(best["TrainLoss"]) and pd.notna(last["TrainLoss"]) else None
            ),
            "DeltaEval(Last-Best)": last["EvalLoss"] - best["EvalLoss"],
            "PossibleOverfitting": overfit_flag,
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("Split")
    print("\nOverfitting check by split:")
    print(summary_df.to_string(index=False))

    print("\nMean train_loss (at eval points):", round(epoch_df["TrainLoss"].dropna().mean(), 4))
    print("Mean eval_loss:", round(epoch_df["EvalLoss"].mean(), 4))
    print("Mean eval_accuracy:", round(epoch_df["EvalAccuracy"].mean(), 4))
    print("Folds flagged as possible overfitting:", int(summary_df["PossibleOverfitting"].sum()), "/", len(summary_df))
else:
    print("No eval logs found under fold directories.")
    print("Checked under:", os.path.join(latest, "cv"))
    print("Try re-running CV (section 3), then this cell.")

Train/validation metrics per epoch by split:
  Split  Epoch  TrainLoss  EvalLoss  EvalAccuracy
split_1    1.0   0.525633  0.502771      0.765677
split_1    2.0   0.357095  0.474508      0.784378
split_2    1.0   0.557724  0.495439      0.774477
split_2    2.0   0.365135  0.469728      0.788779
split_3    1.0   0.501467  0.507615      0.763216
split_3    2.0   0.428720  0.470911      0.789648
split_4    1.0   0.531211  0.517236      0.752203
split_4    2.0   0.404152  0.478230      0.781938
split_5    1.0   0.535883  0.473666      0.783040
split_5    2.0   0.415069  0.461791      0.794053

Overfitting check by split:
  Split  BestEpoch  BestTrainLoss  BestEvalLoss  LastEpoch  LastTrainLoss  LastEvalLoss  DeltaTrain(Last-Best)  DeltaEval(Last-Best)  PossibleOverfitting
split_1        2.0       0.357095      0.474508        2.0       0.357095      0.474508                    0.0                   0.0                False
split_2        2.0       0.365135      0.469728        2.0       0.3

## 5. Train final model on full training set and zip it

In [ ]:
import os, glob, shutil, subprocess

# Fallback hyperparameters if section 2.5 wasn't run
try:
    _ = EPOCHS
except NameError:
    EPOCHS = None
try:
    _ = LEARNING_RATE
except NameError:
    LEARNING_RATE = None
try:
    _ = BATCH_SIZE
except NameError:
    BATCH_SIZE = None
try:
    _ = WEIGHT_DECAY
except NameError:
    WEIGHT_DECAY = None

# Build command with same hyperparameters used in CV
cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "full",
    "--model", MODEL_KEY,
    "--output_root", "./outputs",
    "--seed", "42"
]

if EPOCHS is not None:
    cmd.extend(["--epochs", str(EPOCHS)])
if LEARNING_RATE is not None:
    cmd.extend(["--lr", str(LEARNING_RATE)])
if BATCH_SIZE is not None:
    cmd.extend(["--batch_size", str(BATCH_SIZE)])
if WEIGHT_DECAY is not None:
    cmd.extend(["--weight_decay", str(WEIGHT_DECAY)])

print("Running full training with same hyperparameters as CV:")
print(" ".join(cmd))
print()

# Stream child process output live in notebook
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

# Find newest full-training model folder
model_dirs = sorted(glob.glob(f"./outputs/{MODEL_KEY}_*/model", recursive=True))
if not model_dirs:
    raise FileNotFoundError(f"No trained model folder found under ./outputs/{MODEL_KEY}_*/model")

model_dir = model_dirs[-1]
zip_base = f"{MODEL_KEY}_trained"
zip_file = zip_base + ".zip"

print("Using model dir:", model_dir)
shutil.make_archive(zip_base, "zip", model_dir)
print("Created zip:", zip_file)
print("Zip exists:", os.path.exists(zip_file), "| Size (MB):", round(os.path.getsize(zip_file) / 1024 / 1024, 2))

# Save the full path for section 6
ZIP_PATH = os.path.abspath(zip_file)
print(f"Zip file saved at: {ZIP_PATH}")

Running full training with same hyperparameters as CV:
python src/pipeline/run_pipeline.py --mode full --model sbert --output_root ./outputs --seed 42 --lr 4e-05

Training model preset: sbert
Backbone: sentence-transformers/all-mpnet-base-v2
Loaded 4542 rows from data/hippocorpus/hippocorpus_training_truncated.csv

Map: 100%|██████████| 4542/4542 [00:00<00:00, 7637.01 examples/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3294.31it/s, Materializing param=mpnet.encoder.relative_attention_bias.weight]
MPNetForSequenceClassification LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Note

## 6. Copy trained model to Google Drive
Use this after full training if you want the trained model zip available outside the Colab runtime.

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")
drive_dir = "/content/drive/MyDrive"

# Copy trained model zip from section 5
model_zip_src = ZIP_PATH if "ZIP_PATH" in globals() else f"{MODEL_KEY}_trained.zip"
assert os.path.exists(model_zip_src), f"Missing file: {model_zip_src}. Run section 5 first."
model_zip_dst = f"{drive_dir}/{MODEL_KEY}_trained.zip"
shutil.copy(model_zip_src, model_zip_dst)
print(f"Copied model zip: {model_zip_src} -> {model_zip_dst}")
print(f"Model zip size: {round(os.path.getsize(model_zip_dst) / 1024 / 1024, 2)} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied model zip: /content/deceptionClassifierTraining/sbert_retrained.zip -> /content/drive/MyDrive/sbert_retrained.zip
Model zip size: 386.78 MB


## 7. Optional: run evaluation in Colab and copy results to Drive
Use this only if you want to evaluate inside Colab and keep those outputs. In most cases, evaluation is better run locally in VS Code so the results are written directly into your local project. If you do evaluate in Colab, run the next code cell first, then run the Drive upload cell after it.

In [ ]:
import glob
import os
import subprocess

# Evaluation output mode: combined, per-model, or both
EVAL_LABELED_OUTPUT = "combined"

# Prefer the model path from section 5; fallback to latest saved model directory
if "model_dir" in globals() and os.path.isdir(model_dir):
    model_dir_to_eval = model_dir
else:
    candidates = sorted(glob.glob(f"./outputs/{MODEL_KEY}_*/model", recursive=True))
    if not candidates:
        raise FileNotFoundError("No model directory found. Run section 5 first.")
    model_dir_to_eval = candidates[-1]

results_dir = os.path.abspath("./results")
os.makedirs(results_dir, exist_ok=True)

cmd = [
    "python", "src/pipeline/run_pipeline.py",
    "--mode", "eval",
    "--model_dir", model_dir_to_eval,
    "--results_dir", results_dir,
    "--labeled_output", EVAL_LABELED_OUTPUT,
]

print("Running evaluation with command:")
print(" ".join(cmd))
print(f"Labeled output mode: {EVAL_LABELED_OUTPUT}")
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    )
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

print("\nEvaluation finished.")
print(f"Results directory: {results_dir}")
print("Generated files:")
for path in sorted(glob.glob(os.path.join(results_dir, "*"))):
    print(" ", os.path.basename(path))

In [ ]:
import os
import shutil

drive_dir = "/content/drive/MyDrive"
results_dir = os.path.abspath("./results")

if not os.path.isdir(results_dir) or not os.listdir(results_dir):
    raise FileNotFoundError(
        "No evaluation results found under ./results. Run the evaluation cell first, or evaluate locally in VS Code instead."
    )

results_zip_base = f"{MODEL_KEY}_results"
results_zip_path = shutil.make_archive(results_zip_base, "zip", results_dir)
results_zip_dst = f"{drive_dir}/{MODEL_KEY}_results.zip"
shutil.copy(results_zip_path, results_zip_dst)
print(f"Copied results zip: {results_zip_path} -> {results_zip_dst}")
print(f"Results zip size: {round(os.path.getsize(results_zip_dst) / 1024 / 1024, 2)} MB")